# MASI Colab Smoke Test (Fresh Clone)

This notebook is the safest Colab bootstrap path for MASI smoke testing.

It does all of the following:
- deletes any old `/content/MASI` clone,
- clones the latest repo again,
- installs dependencies into the active Colab runtime,
- downloads the bounded CSJ review slice,
- runs the one-click smoke pipeline,
- checks whether the expected output manifest exists before trying to open it.

The notebook itself does not contain MASI training logic. It only orchestrates the checked-in repository scripts so the Colab run stays consistent with the repo.

## Configure Repo

Set your repo URL and branch below before running all cells.

In [2]:
REPO_URL = "https://github.com/pradyunuydarp/MASI.git"
BRANCH = "main"
REPO_DIR = "/content/MASI"
STORAGE_ROOT = "/content/masi_storage"
CONFIG_PATH = "configs/masi_train_csj_medium_colab.json"
RUN_NAME = "amazon_csj_medium_colab_train"

## Fresh Clone

This cell deletes any older Colab checkout so you always run against the latest pushed repo state.

In [3]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git rev-parse --short HEAD
!ls -lah scripts/train_masi.py "$CONFIG_PATH" notebooks/03_colab_smoke_test.ipynb

/content
Cloning into '/content/MASI'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 128 (delta 37), reused 112 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 311.08 KiB | 1.87 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/MASI
28cf7db
-rw-r--r-- 1 root root 2.1K Apr 19 07:37 configs/masi_train_csj_medium_colab.json
-rw-r--r-- 1 root root 6.1K Apr 19 07:37 notebooks/03_colab_smoke_test.ipynb
-rw-r--r-- 1 root root  13K Apr 19 07:37 scripts/train_masi.py


## Install Dependencies

In [4]:
%cd "$REPO_DIR"
import sys
print("Notebook Python:", sys.executable)
%pip install --upgrade pip
%pip install -e ".[recommender]"

/content/MASI
Notebook Python: /usr/bin/python3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/MASI
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 34.4 MB/s  0:00:00 eta 0:00:01
  Building editable for masi (pyproject.toml) ... done
  Created wheel for masi: filename=masi-0.1.0-0.editable-py3-none-any.whl size=7877 sha256=9fa0a8a4d6db47223b18ec20e168004ff6b5a6e2c624816e96055f3915ae206e
  Stored in directory: /tmp/pip-ephem-wheel-cache-eq_aw90m/wheels/5b/20/38/b6d50dcfbc84dbc1346a59162946e6fc870d54973538735a91
Successfully built masi
  Attempting uninstall: numpy
    F

## Prepare Folders

In [5]:
%cd "$REPO_DIR"
!mkdir -p "$STORAGE_ROOT"
!mkdir -p "$STORAGE_ROOT/data/raw/amazon_reviews_2023"

/content/MASI


## Download Bounded CSJ Slice

This downloads the medium Colab review slice into `STORAGE_ROOT`, matching the one-click config.

In [6]:
%cd "$REPO_DIR"
!PYTHONPATH=src python scripts/download_amazon_csj_dataset.py \
  --output-dir "$STORAGE_ROOT/data/raw/amazon_reviews_2023" \
  --review-range-end 6442450943
!ls -lh "$STORAGE_ROOT/data/raw/amazon_reviews_2023"

/content/MASI
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1383  100  1383    0     0   8470      0 --:--:-- --:--:-- --:--:--  8484
100 6144M  100 6144M    0     0   143M      0  0:00:42  0:00:42 --:--:--  192M
Downloaded reviews to /content/masi_storage/data/raw/amazon_reviews_2023/Clothing_Shoes_and_Jewelry.jsonl
total 6.1G
-rw-r--r-- 1 root root 6.0G Apr 19 07:38 Clothing_Shoes_and_Jewelry.jsonl


## Run Medium Pipeline

In [7]:
%cd "$REPO_DIR"
!PYTHONPATH=src python scripts/train_masi.py \
  --config "$CONFIG_PATH" \
  --storage-root "$STORAGE_ROOT"

/content/MASI
config.json: 4.19kB [00:00, 12.9MB/s]
pytorch_model.bin: 100% 605M/605M [00:03<00:00, 186MB/s] 
Loading weights:  55% 219/398 [00:00<00:00, 1198.60it/s, Materializing param=vision_model.encoder.layers.1.layer_norm1.weight]       
Loading weights: 100% 398/398 [00:00<00:00, 788.71it/s, Materializing param=visual_projection.weight]                                

model.safetensors:   0% 0.00/605M [00:00<?, ?B/s]CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

model.safetensors:   1% 8.39M/605M [00:00<00:14, 41.8MB/s]
model.safetensors:   8% 50.0M/605M [00:00<00:06, 87.6MB/s]
model.safetensors:  17% 101M/605M [00:01<00:03, 143

## Inspect Output Files

In [8]:
!find "$STORAGE_ROOT" -maxdepth 4 -type f | sort

/content/masi_storage/data/processed/amazon_csj_medium_colab/metadata.slice.jsonl
/content/masi_storage/data/raw/amazon_reviews_2023/Clothing_Shoes_and_Jewelry.jsonl
/content/masi_storage/outputs/amazon_csj_medium_colab_train/phase12_tokens/fused_semantic_ids.jsonl
/content/masi_storage/outputs/amazon_csj_medium_colab_train/phase12_tokens/masi_token_summary.json
/content/masi_storage/outputs/amazon_csj_medium_colab_train/phase3_experiment/experiment_summary.json
/content/masi_storage/outputs/amazon_csj_medium_colab_train/resolved_configs/experiment.json
/content/masi_storage/outputs/amazon_csj_medium_colab_train/resolved_configs/token_build.json
/content/masi_storage/outputs/amazon_csj_medium_colab_train/run_manifest.json


## Load Manifest Safely

This cell checks whether the medium run actually produced the manifest before trying to open it.

In [9]:
import json
from pathlib import Path

manifest_path = Path(STORAGE_ROOT) / "outputs" / RUN_NAME / "run_manifest.json"
print("Run manifest path:", manifest_path)
print("Exists:", manifest_path.exists())

if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as handle:
        manifest = json.load(handle)
    print("Environment:", manifest["environment"])
    print("Token summary path:", manifest["token_summary_path"])
    print("Experiment summary path:", manifest["experiment_summary_path"])
else:
    print("Run did not finish successfully. Re-check the previous training cell output.")

Run manifest path: /content/masi_storage/outputs/amazon_csj_medium_colab_train/run_manifest.json
Exists: True
Environment: colab
Token summary path: /content/masi_storage/outputs/amazon_csj_medium_colab_train/phase12_tokens/masi_token_summary.json
Experiment summary path: /content/masi_storage/outputs/amazon_csj_medium_colab_train/phase3_experiment/experiment_summary.json


## Load Experiment Summary Safely

In [10]:
import json
from pathlib import Path

summary_path = Path(STORAGE_ROOT) / "outputs" / RUN_NAME / "phase3_experiment" / "experiment_summary.json"
print("Summary path:", summary_path)
print("Exists:", summary_path.exists())

if summary_path.exists():
    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)
    high_signal = {
        "mlm_status": summary.get("mlm_status"),
        "generative_finetuning_status": summary.get("generative_finetuning_status"),
        "num_items": summary.get("num_items"),
        "num_train_examples": summary.get("num_train_examples"),
        "num_mlm_examples": summary.get("num_mlm_examples"),
        "warm_metrics": summary.get("warm_metrics"),
        "cold_metrics": summary.get("cold_metrics"),
    }
    print(json.dumps(high_signal, indent=2))
else:
    print("Experiment summary is missing because the medium run did not complete.")

Summary path: /content/masi_storage/outputs/amazon_csj_medium_colab_train/phase3_experiment/experiment_summary.json
Exists: True
{
  "mlm_status": "trained",
  "generative_finetuning_status": "trained",
  "num_items": 989,
  "num_train_examples": 7237,
  "num_mlm_examples": 1978,
  "warm_metrics": {
    "avg_latency_ms": 1956.8808136532816,
    "coverage@10": 0.4327603640040445,
    "hr@10": 0.0830945558739255,
    "ndcg@10": 0.05997318439786636,
    "num_examples": 349
  },
  "cold_metrics": {
    "avg_latency_ms": 1942.3460762469056,
    "coverage@10": 0.268958543983822,
    "hr@10": 0.0,
    "ndcg@10": 0.0,
    "num_examples": 81
  }
}
